In [1]:
!gdown --folder "https://drive.google.com/drive/folders/1uMBAahM95kIzD17_k9S7lcqAkbJggHb4"

Retrieving folder contents
Processing file 1hEdQ3Gy_pSjjqkEHCH6qGZwofY8SQGmi sample_submission.csv
Processing file 1eSxQO14l_3MC6gKOUCHFi9HQW6_LvGBO test_apps.csv
Processing file 1-fM8nFRcT2t_mTyP4nvPvzpc6WYB-JXm train_apps.csv
Processing file 1PyOcSjZ4ClzLYkmThY2fPbyVlGlrfdJj Альфа-Банк х МФТИ. Отклик на кредитный оффер.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1hEdQ3Gy_pSjjqkEHCH6qGZwofY8SQGmi
To: /content/Задача 1 Отклик на кредитный оффер/sample_submission.csv
100% 540k/540k [00:00<00:00, 99.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eSxQO14l_3MC6gKOUCHFi9HQW6_LvGBO
To: /content/Задача 1 Отклик на кредитный оффер/test_apps.csv
100% 12.4M/12.4M [00:00<00:00, 33.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-fM8nFRcT2t_mTyP4nvPvzpc6WYB-JXm
To: /content/Задача 1 Отклик на кредитный оффер/train_apps.csv
100% 45.9M/45.9M [00:00<00:00, 72.

## 1. Установка зависимостей

In [2]:
!pip install catboost lightgbm xgboost optuna tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 22.1 MB/s eta 0:00:00


## 2. Импорты и настройки

In [3]:
import warnings, os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm
warnings.filterwarnings("ignore")

DATADIR  = "/content/data"
TARGET   = "target_value"
ID_COL   = "front_id"
SEED     = 42
N_FOLDS  = 5
DROP_FEATURES          = {"decision_ordinal"}
CATEGORICAL_CANDIDATES = ["db_group_last", "fl_adminarea"]

## 3. Feature Engineering

In [5]:
def safe_ratio(num, den):
    return num / den.replace(0, np.nan)


def add_features(df):
    df = df.copy()
    cols = set(df.columns)

    if "decision_day" in cols:
        dt = pd.to_datetime(df["decision_day"], errors="coerce")
        df["decision_ordinal"] = (dt - pd.Timestamp("2020-01-01")).dt.days
        df["decision_month"]   = dt.dt.month
        df["decision_dow"]     = dt.dt.dayofweek
        df["decision_dom"]     = dt.dt.day
        df = df.drop(columns=["decision_day"])

    if {"offered_rate", "cb_rate"} <= cols:
        df["rate_spread"] = df["offered_rate"] - df["cb_rate"]
        df["rate_ratio"]  = safe_ratio(df["offered_rate"], df["cb_rate"])

    if {"overdraft_limit_max", "overdraft_limit_min"} <= cols:
        df["overdraft_limit_range"] = df["overdraft_limit_max"] - df["overdraft_limit_min"]
        df["overdraft_limit_mid"]   = (df["overdraft_limit_max"] + df["overdraft_limit_min"]) / 2

    if {"loan_amount_last", "overdraft_limit_max"} <= cols:
        df["loan_vs_odmax"] = safe_ratio(df["loan_amount_last"], df["overdraft_limit_max"])
    if {"loan_amount_last", "overdraft_limit_mid"} <= cols:
        df["loan_vs_odmid"] = safe_ratio(df["loan_amount_last"], df["overdraft_limit_mid"])
    if {"loan_amount_last", "balance_rur_amt_30_min"} <= cols:
        df["loan_vs_balance"] = safe_ratio(df["loan_amount_last"], df["balance_rur_amt_30_min"])

    if {"sum_deb_ul_30", "sum_deb_ul_90"} <= cols:
        df["sum_deb_ul_ratio_30_90"] = safe_ratio(df["sum_deb_ul_30"], df["sum_deb_ul_90"])
        df["sum_deb_ul_60"]          = df["sum_deb_ul_90"] - df["sum_deb_ul_30"]
    if {"cnt_deb_ul_ip_30", "cnt_deb_ul_ip_90"} <= cols:
        df["cnt_deb_ul_ip_ratio_30_90"] = safe_ratio(df["cnt_deb_ul_ip_30"], df["cnt_deb_ul_ip_90"])

    if {"cnt_cred_loan_90", "cnt_deb_loan_90"} <= cols:
        df["net_loan_flow_90"] = df["cnt_cred_loan_90"] - df["cnt_deb_loan_90"]

    if {"loan_amount_last", "overdraft_limit_min", "overdraft_limit_max"} <= cols:
        rng = (df["overdraft_limit_max"] - df["overdraft_limit_min"]).replace(0, np.nan)
        df["loan_pos_in_corridor"] = (df["loan_amount_last"] - df["overdraft_limit_min"]) / rng

    if {"sum_deb_ul_90", "cnt_deb_ul_ip_90"} <= cols:
        df["avg_transfer_ul_90"] = safe_ratio(df["sum_deb_ul_90"], df["cnt_deb_ul_ip_90"])
    if {"sum_deb_ul_30", "cnt_deb_ul_ip_30"} <= cols:
        df["avg_transfer_ul_30"] = safe_ratio(df["sum_deb_ul_30"], df["cnt_deb_ul_ip_30"])

    if {"cnt_deb_ul_ip_30", "cnt_deb_ul_ip_90"} <= cols:
        df["cnt_ul_ip_accel"] = df["cnt_deb_ul_ip_30"] - df["cnt_deb_ul_ip_90"] / 3.0
    if {"sum_deb_ul_30", "sum_deb_ul_90"} <= cols:
        df["sum_ul_accel"] = df["sum_deb_ul_30"] - df["sum_deb_ul_90"] / 3.0

    if {"count_all_corp_dashboard_events", "p75_time_spent_minutes"} <= cols:
        df["events_per_minute"] = safe_ratio(
            df["count_all_corp_dashboard_events"], df["p75_time_spent_minutes"])

    if {"offered_rate", "loan_amount_last"} <= cols:
        df["rate_x_amount"] = df["offered_rate"] * np.log1p(df["loan_amount_last"].abs())

    if {"balance_rur_amt_30_min", "overdraft_limit_max"} <= cols:
        df["balance_vs_odmax"] = safe_ratio(df["balance_rur_amt_30_min"], df["overdraft_limit_max"])
    if {"sum_deb_investment_90", "sum_deb_ul_90"} <= cols:
        df["investment_share"] = safe_ratio(
            df["sum_deb_investment_90"], df["sum_deb_ul_90"].abs() + 1)

    for c in ["loan_amount_last","overdraft_limit_min","overdraft_limit_max",
              "sum_deb_ul_90","sum_deb_ul_30","balance_rur_amt_30_min",
              "sum_deb_investment_90","days_from_authperson_registration"]:
        if c in cols:
            df[f"{c}_log1p"] = np.sign(df[c]) * np.log1p(df[c].abs())

    for c in ["loan_rev_max_start_non_fin","loan_rev_min_start_fin","app_term_mean_360",
              "overdraft_app_term_max_360","days_from_authperson_registration",
              "fl_hdb_bki_total_active_products","p75_time_spent_minutes",
              "sum_deb_investment_90","balance_rur_amt_30_min","corp_credit_products"]:
        if c in cols:
            df[f"{c}_isna"] = df[c].isna().astype("int8")

    src_cols = [c for c in cols if c not in {ID_COL, TARGET, "decision_day"}]
    if src_cols:
        df["row_nulls"] = df[src_cols].isna().sum(axis=1)

    return df


def build_design_matrix(train, test):
    train = add_features(train)
    test  = add_features(test)
    drop_cols = {ID_COL, TARGET} | DROP_FEATURES
    feats = [c for c in train.columns if c not in drop_cols and c in test.columns]
    cat_cols = [c for c in CATEGORICAL_CANDIDATES if c in feats]
    for c in feats:
        if c not in cat_cols and not pd.api.types.is_numeric_dtype(train[c]):
            cat_cols.append(c)
    for c in cat_cols:
        train[c] = train[c].astype("object").where(train[c].notna(), "NA").astype(str)
        test[c]  = test[c].astype("object").where(test[c].notna(), "NA").astype(str)
    return train, test, feats, cat_cols


def target_encode(train, test, y, col, folds, smoothing=30.0):
    global_mean = y.mean()
    def make_map(cats, ys):
        d = pd.DataFrame({"c": cats, "y": ys})
        agg = d.groupby("c")["y"].agg(["mean","count"])
        return (agg["count"] * agg["mean"] + smoothing * global_mean) / (agg["count"] + smoothing)
    oof  = np.full(len(train), global_mean, dtype=float)
    cats = train[col].astype(str).values
    for tr, va in folds:
        enc = make_map(cats[tr], y.values[tr])
        oof[va] = pd.Series(cats[va]).map(enc).fillna(global_mean).values
    enc_full = make_map(cats, y.values)
    test_enc = test[col].astype(str).map(enc_full).fillna(global_mean).values
    return oof, test_enc


def to_rank(x):
    return pd.Series(x).rank(pct=True).values

## 4. Модели

In [6]:
def train_catboost(X, y, X_test, feats, cat_cols, folds, override=None):
    from catboost import CatBoostClassifier, Pool
    oof, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
    cat_idx = [feats.index(c) for c in cat_cols]
    params = dict(iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=6.0,
                  loss_function="Logloss", eval_metric="AUC", random_seed=SEED,
                  early_stopping_rounds=200, verbose=0, allow_writing_files=False)
    if override:
        params.update(override)
    aucs = []
    for fold, (tr, va) in enumerate(tqdm(folds, desc="CatBoost", leave=False)):
        m = CatBoostClassifier(**params)
        m.fit(Pool(X.iloc[tr][feats], y.iloc[tr], cat_features=cat_idx),
              eval_set=Pool(X.iloc[va][feats], y.iloc[va], cat_features=cat_idx))
        oof[va] = m.predict_proba(X.iloc[va][feats])[:, 1]
        test_pred += m.predict_proba(X_test[feats])[:, 1] / len(folds)
        aucs.append(roc_auc_score(y.iloc[va], oof[va]))
    oof_auc = roc_auc_score(y, oof)
    print(f"  CatBoost  folds {[f'{a:.4f}' for a in aucs]}  →  OOF {oof_auc:.5f}")
    return oof, test_pred


def train_lightgbm(X, y, X_test, feats, cat_cols, folds, override=None):
    import lightgbm as lgb
    oof, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
    Xc, Xt = X.copy(), X_test.copy()
    for c in cat_cols:
        Xc[c] = Xc[c].astype("category")
        Xt[c] = pd.Categorical(Xt[c], categories=Xc[c].cat.categories)
    params = dict(objective="binary", metric="auc", learning_rate=0.02, num_leaves=63,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                  min_child_samples=50, lambda_l1=1.0, lambda_l2=2.0,
                  n_estimators=5000, random_state=SEED, n_jobs=-1, verbose=-1)
    if override:
        params.update(override)
    aucs = []
    for fold, (tr, va) in enumerate(tqdm(folds, desc="LightGBM", leave=False)):
        m = lgb.LGBMClassifier(**params)
        m.fit(Xc.iloc[tr][feats], y.iloc[tr],
              eval_set=[(Xc.iloc[va][feats], y.iloc[va])],
              categorical_feature=cat_cols,
              callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
        oof[va] = m.predict_proba(Xc.iloc[va][feats])[:, 1]
        test_pred += m.predict_proba(Xt[feats])[:, 1] / len(folds)
        aucs.append(roc_auc_score(y.iloc[va], oof[va]))
    oof_auc = roc_auc_score(y, oof)
    print(f"  LightGBM  folds {[f'{a:.4f}' for a in aucs]}  →  OOF {oof_auc:.5f}")
    return oof, test_pred


def train_xgboost(X, y, X_test, feats, cat_cols, folds):
    import xgboost as xgb
    oof, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
    Xc, Xt = X.copy(), X_test.copy()
    for c in cat_cols:
        Xc[c] = Xc[c].astype("category")
        Xt[c] = pd.Categorical(Xt[c], categories=Xc[c].cat.categories)
    params = dict(objective="binary:logistic", eval_metric="auc", learning_rate=0.03,
                  max_depth=6, subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                  reg_lambda=3.0, reg_alpha=1.0, n_estimators=4000, tree_method="hist",
                  enable_categorical=True, random_state=SEED, n_jobs=-1,
                  early_stopping_rounds=200)
    aucs = []
    for fold, (tr, va) in enumerate(tqdm(folds, desc="XGBoost ", leave=False)):
        m = xgb.XGBClassifier(**params)
        m.fit(Xc.iloc[tr][feats], y.iloc[tr],
              eval_set=[(Xc.iloc[va][feats], y.iloc[va])], verbose=False)
        oof[va] = m.predict_proba(Xc.iloc[va][feats])[:, 1]
        test_pred += m.predict_proba(Xt[feats])[:, 1] / len(folds)
        aucs.append(roc_auc_score(y.iloc[va], oof[va]))
    oof_auc = roc_auc_score(y, oof)
    print(f"  XGBoost   folds {[f'{a:.4f}' for a in aucs]}  →  OOF {oof_auc:.5f}")
    return oof, test_pred

## 5. Диагностика сдвига

In [7]:
def adversarial_validation(train, test, feats, cat_cols):
    from catboost import CatBoostClassifier, Pool
    Xa = pd.concat([train[feats], test[feats]], ignore_index=True)
    ya = np.r_[np.zeros(len(train)), np.ones(len(test))]
    cat_idx = [feats.index(c) for c in cat_cols]
    rng = np.random.RandomState(SEED)
    idx = rng.permutation(len(Xa))
    cut = int(len(Xa) * 0.8)
    tr, va = idx[:cut], idx[cut:]
    m = CatBoostClassifier(iterations=400, depth=5, learning_rate=0.05,
                           eval_metric="AUC", random_seed=SEED,
                           verbose=0, allow_writing_files=False)
    m.fit(Pool(Xa.iloc[tr], ya[tr], cat_features=cat_idx),
          eval_set=Pool(Xa.iloc[va], ya[va], cat_features=cat_idx))
    auc = roc_auc_score(ya[va], m.predict_proba(Xa.iloc[va])[:, 1])
    imp = pd.Series(m.feature_importances_, index=feats).sort_values(ascending=False).head(10)
    shift = "норм" if auc < 0.65 else "умеренный" if auc < 0.8 else "сильный сдвиг"
    print(f"  Adversarial AUC: {auc:.4f}  ({shift})")
    print("  Топ признаков сдвига:")
    for f, v in imp.items():
        print(f"    {f:<38} {v:5.1f}%")


def time_holdout_eval(train, y, feats, cat_cols, order, frac=0.2):
    from catboost import CatBoostClassifier, Pool
    cut = int(len(order) * (1 - frac))
    tr_idx, va_idx = order[:cut], order[cut:]
    cat_idx = [feats.index(c) for c in cat_cols]
    m = CatBoostClassifier(iterations=4000, depth=5, learning_rate=0.025,
                           l2_leaf_reg=10.0, eval_metric="AUC", random_seed=SEED,
                           early_stopping_rounds=200, verbose=0, allow_writing_files=False)
    m.fit(Pool(train.iloc[tr_idx][feats], y.iloc[tr_idx], cat_features=cat_idx),
          eval_set=Pool(train.iloc[va_idx][feats], y.iloc[va_idx], cat_features=cat_idx))
    auc = roc_auc_score(y.iloc[va_idx], m.predict_proba(train.iloc[va_idx][feats])[:, 1])
    print(f"  Time-split AUC (последние {int(frac*100)}%): {auc:.5f}  (ориентир на лидерборд)")
    return auc

## 6. Optuna-тюнинг

In [8]:
def tune_models(train, y, feats, cat_cols, order, n_trials=35, frac=0.2):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    cut = int(len(order) * (1 - frac))
    tr_idx, va_idx = order[:cut], order[cut:]
    cat_idx = [feats.index(c) for c in cat_cols]
    best = {}

    from catboost import CatBoostClassifier, Pool
    tr_pool = Pool(train.iloc[tr_idx][feats], y.iloc[tr_idx], cat_features=cat_idx)
    va_pool = Pool(train.iloc[va_idx][feats], y.iloc[va_idx], cat_features=cat_idx)
    def cb_obj(trial):
        p = dict(iterations=5000, loss_function="Logloss", eval_metric="AUC",
                 random_seed=SEED, early_stopping_rounds=150, verbose=0, allow_writing_files=False,
                 learning_rate=trial.suggest_float("lr",    0.01, 0.08, log=True),
                 depth=           trial.suggest_int(  "depth", 4, 8),
                 l2_leaf_reg=     trial.suggest_float("l2",    1.0, 20.0, log=True),
                 random_strength= trial.suggest_float("rs",    0.0, 5.0),
                 bagging_temperature=trial.suggest_float("bt", 0.0, 2.0),
                 border_count=    trial.suggest_int(  "bc",    64, 254))
        m = CatBoostClassifier(**p)
        m.fit(tr_pool, eval_set=va_pool)
        return roc_auc_score(y.iloc[va_idx], m.predict_proba(train.iloc[va_idx][feats])[:, 1])
    s = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    s.optimize(cb_obj, n_trials=n_trials, show_progress_bar=True)
    print(f"  CatBoost  best time-AUC = {s.best_value:.5f}")
    p = s.best_params
    best["catboost"] = dict(learning_rate=p["lr"], depth=p["depth"], l2_leaf_reg=p["l2"],
                            random_strength=p["rs"], bagging_temperature=p["bt"],
                            border_count=p["bc"])

    import lightgbm as lgb
    Xtr = train.iloc[tr_idx][feats].copy()
    Xva = train.iloc[va_idx][feats].copy()
    for c in cat_cols:
        Xtr[c] = Xtr[c].astype("category")
        Xva[c] = pd.Categorical(Xva[c], categories=Xtr[c].cat.categories)
    def lgb_obj(trial):
        p = dict(objective="binary", metric="auc", verbose=-1, n_estimators=5000,
                 random_state=SEED, n_jobs=-1, bagging_freq=1,
                 learning_rate=    trial.suggest_float("lr", 0.01, 0.08, log=True),
                 num_leaves=       trial.suggest_int(  "nl", 20, 120),
                 max_depth=        trial.suggest_int(  "md", 4, 8),
                 feature_fraction= trial.suggest_float("ff", 0.5, 1.0),
                 bagging_fraction= trial.suggest_float("bf", 0.5, 1.0),
                 min_child_samples=trial.suggest_int(  "mc", 20, 200),
                 lambda_l1=        trial.suggest_float("l1", 0.01, 10.0, log=True),
                 lambda_l2=        trial.suggest_float("l2", 0.01, 10.0, log=True))
        m = lgb.LGBMClassifier(**p)
        m.fit(Xtr[feats], y.iloc[tr_idx], eval_set=[(Xva[feats], y.iloc[va_idx])],
              categorical_feature=cat_cols,
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        return roc_auc_score(y.iloc[va_idx], m.predict_proba(Xva[feats])[:, 1])
    s2 = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    s2.optimize(lgb_obj, n_trials=n_trials, show_progress_bar=True)
    print(f"  LightGBM  best time-AUC = {s2.best_value:.5f}")
    p2 = s2.best_params
    best["lightgbm"] = dict(learning_rate=p2["lr"], num_leaves=p2["nl"], max_depth=p2["md"],
                            feature_fraction=p2["ff"], bagging_fraction=p2["bf"], bagging_freq=1,
                            min_child_samples=p2["mc"], lambda_l1=p2["l1"], lambda_l2=p2["l2"])
    return best

## 5. Диагностика сдвига

In [9]:
def adversarial_validation(train, test, feats, cat_cols):
    from catboost import CatBoostClassifier, Pool
    Xa = pd.concat([train[feats], test[feats]], ignore_index=True)
    ya = np.r_[np.zeros(len(train)), np.ones(len(test))]
    cat_idx = [feats.index(c) for c in cat_cols]
    rng = np.random.RandomState(SEED)
    idx = rng.permutation(len(Xa))
    cut = int(len(Xa) * 0.8)
    tr, va = idx[:cut], idx[cut:]
    m = CatBoostClassifier(iterations=400, depth=5, learning_rate=0.05,
                           eval_metric="AUC", random_seed=SEED,
                           verbose=0, allow_writing_files=False)
    m.fit(Pool(Xa.iloc[tr], ya[tr], cat_features=cat_idx),
          eval_set=Pool(Xa.iloc[va], ya[va], cat_features=cat_idx))
    auc = roc_auc_score(ya[va], m.predict_proba(Xa.iloc[va])[:, 1])
    imp = pd.Series(m.feature_importances_, index=feats).sort_values(ascending=False).head(10)
    shift = "норм" if auc < 0.65 else "умеренный" if auc < 0.8 else "сильный сдвиг"
    print(f"  Adversarial AUC: {auc:.4f}  ({shift})")
    print("  Топ признаков сдвига:")
    for f, v in imp.items():
        print(f"    {f:<38} {v:5.1f}%")


def time_holdout_eval(train, y, feats, cat_cols, order, frac=0.2):
    from catboost import CatBoostClassifier, Pool
    cut = int(len(order) * (1 - frac))
    tr_idx, va_idx = order[:cut], order[cut:]
    cat_idx = [feats.index(c) for c in cat_cols]
    m = CatBoostClassifier(iterations=4000, depth=5, learning_rate=0.025,
                           l2_leaf_reg=10.0, eval_metric="AUC", random_seed=SEED,
                           early_stopping_rounds=200, verbose=0, allow_writing_files=False)
    m.fit(Pool(train.iloc[tr_idx][feats], y.iloc[tr_idx], cat_features=cat_idx),
          eval_set=Pool(train.iloc[va_idx][feats], y.iloc[va_idx], cat_features=cat_idx))
    auc = roc_auc_score(y.iloc[va_idx], m.predict_proba(train.iloc[va_idx][feats])[:, 1])
    print(f"  Time-split AUC (последние {int(frac*100)}%): {auc:.5f}  (ориентир на лидерборд)")
    return auc

## 6. Запуск

In [11]:
TUNE = True

train_df = pd.read_csv(f"{DATADIR}/train_apps.csv")
test_df  = pd.read_csv(f"{DATADIR}/test_apps.csv")

tr_d = pd.to_datetime(train_df["decision_day"], errors="coerce")
te_d = pd.to_datetime(test_df["decision_day"],  errors="coerce")
time_order = tr_d.reset_index(drop=True).sort_values(kind="mergesort").index.to_numpy()

print(f"  train {train_df.shape}  |  test {test_df.shape}")
print(f"  train {tr_d.min().date()} -> {tr_d.max().date()}")
print(f"  test  {te_d.min().date()} -> {te_d.max().date()}")
print(f"  target rate: {train_df[TARGET].mean():.2%}")

train_df, test_df, feats, cat_cols = build_design_matrix(train_df, test_df)
y = train_df[TARGET].astype(int)
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train_df, y))

for col in cat_cols:
    train_df[f"te_{col}"], test_df[f"te_{col}"] = target_encode(train_df, test_df, y, col, folds)
    feats.append(f"te_{col}")

print(f"\n  признаков: {len(feats)}  |  категориальных: {cat_cols}")

print("\n  Adversarial validation")
adversarial_validation(train_df, test_df, feats, cat_cols)
print("\n  Time-split оценка")
time_holdout_eval(train_df, y, feats, cat_cols, time_order)

best_params = {}
if TUNE:
    print("\n  Optuna тюнинг...")
    best_params = tune_models(train_df, y, feats, cat_cols, time_order)

print("\n  Обучение ансамбля")
oof_cb,  test_cb  = train_catboost(train_df, y, test_df, feats, cat_cols, folds,
                                    override=best_params.get("catboost"))
oof_lgb, test_lgb = train_lightgbm(train_df, y, test_df, feats, cat_cols, folds,
                                    override=best_params.get("lightgbm"))
oof_xgb, test_xgb = train_xgboost(train_df, y, test_df, feats, cat_cols, folds)

oof_blend  = np.mean([to_rank(oof_cb),  to_rank(oof_lgb),  to_rank(oof_xgb)],  axis=0)
test_blend = np.mean([to_rank(test_cb), to_rank(test_lgb), to_rank(test_xgb)], axis=0)
print(f"\n  Ensemble OOF AUC: {roc_auc_score(y, oof_blend):.5f}")

sub = pd.read_csv(f"{DATADIR}/sample_submission.csv")
pred_col = [c for c in sub.columns if c != ID_COL][0]
test_map  = dict(zip(test_df[ID_COL].astype(str),  test_blend))
train_map = dict(zip(train_df[ID_COL].astype(str), oof_blend))
key = sub[ID_COL].astype(str)
sub[pred_col] = key.map(test_map)
n_test = sub[pred_col].notna().sum()
sub[pred_col] = sub[pred_col].fillna(key.map(train_map))
n_train = int(sub[pred_col].notna().sum() - n_test)
if sub[pred_col].isna().any():
    sub[pred_col] = sub[pred_col].fillna(float(pd.Series(test_blend).median()))

out = f"{DATADIR}/submission.csv"
sub.to_csv(out, index=False)
print(f"\n  Saved: {out}  ({n_test} from test + {n_train} OOF from train)")
sub.head()

  train (145241, 28)  |  test (36311, 27)
  train 2024-02-01 -> 2025-06-05
  test  2025-06-05 -> 2025-12-01
  target rate: 6.09%

  признаков: 68  |  категориальных: ['db_group_last', 'fl_adminarea']

  Adversarial validation
  Adversarial AUC: 1.0000  (сильный сдвиг)
  Топ признаков сдвига:
    cb_rate                                 31.6%
    te_db_group_last                        27.1%
    decision_month                          24.9%
    rate_ratio                               6.4%
    rate_spread                              5.0%
    db_group_last                            1.2%
    offered_rate                             1.2%
    te_fl_adminarea                          0.8%
    overdraft_limit_max_log1p                0.5%
    decision_dom                             0.3%

  Time-split оценка
  Time-split AUC (последние 20%): 0.76964  (ориентир на лидерборд)

  Optuna тюнинг...


  0%|          | 0/35 [00:00<?, ?it/s]

  CatBoost  best time-AUC = 0.77177


  0%|          | 0/35 [00:00<?, ?it/s]

  LightGBM  best time-AUC = 0.77174

  Обучение ансамбля


...

LightGBM:   0%|          | 0/5 [00:00<?, ?it/s]

  LightGBM  folds ['0.8319', '0.8340', '0.8433', '0.8395', '0.8298']  →  OOF 0.83566


XGBoost :   0%|          | 0/5 [00:00<?, ?it/s]

  XGBoost   folds ['0.8303', '0.8321', '0.8423', '0.8381', '0.8290']  →  OOF 0.83430

  Ensemble OOF AUC: 0.83671

  Saved: /content/data/submission.csv  (36311 from test + 8721 OOF from train)


,front_id,target_value
0,238459,0.335436
1,151533,0.653484
2,195027,0.737214
3,195034,0.802150
4,129620,0.758489
